In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
import plotly.express as px

%matplotlib inline

In [30]:
# Cleaning data and structuring to import to tableau

In [4]:
df=pd.read_csv('../data/clean/merged_data.csv')

/var/folders/bq/jm95c4z529q6_v32y8vmwlk00000gn/T/ipykernel_58721/1677889000.py:1: DtypeWarning: Columns (0: Variation) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('../data/clean/merged_data.csv')


In [7]:
df['Variation'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [8]:
steps_map={'start':0, 'step_1':1, 'step_2':2, 'step_3':3, 'confirm':4}
df['step_num']= df['process_step'].map(steps_map)
df.step_num.value_counts()

step_num
0    234999
1    162797
2    132750
3    111589
4    102506
Name: count, dtype: int64

In [9]:
df=df.sort_values(['client_id','visit_id', 'date_time'], ascending=True)
df.head()

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,step_num
283823,283823,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,0
283822,283822,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,1
283821,283821,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,2
283820,283820,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,3
283819,283819,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,4


In [36]:
def bool_error(series):
    return (series.diff() <= 0)

In [37]:
df['error'] = df.groupby(['client_id','visit_id','Variation'])['step_num'].transform(bool_error)
df.head(10)

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,step_num,error,duration,duration_sec
283823,283823,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,0,NaN,NaT,NaN
283822,283822,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,1,NaN,0 days 00:00:09,9.0
283821,283821,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,2,NaN,0 days 00:00:46,46.0
283820,283820,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,3,NaN,0 days 00:01:34,94.0
283819,283819,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,4,NaN,0 days 00:01:04,64.0
620520,620520,336,64757908_3400128256,649044751_80905125055_554468,start,2017-06-01 07:26:55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaT,NaN
620397,620397,336,64757908_3400128256,649044751_80905125055_554468,start,2017-06-01 07:42:43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0 days 00:15:48,948.0
439053,439053,546,475037402_89828530214,731811517_9330176838_94847,start,2017-06-17 10:03:29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaT,NaN
439052,439052,546,475037402_89828530214,731811517_9330176838_94847,step_1,2017-06-17 10:03:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,0 days 00:00:10,10.0
439051,439051,546,475037402_89828530214,731811517_9330176838_94847,step_2,2017-06-17 10:03:52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,0 days 00:00:13,13.0


In [41]:
df.error.isna().sum()

np.int64(427406)

In [43]:
df['error'] = df['error'].fillna(False)

In [44]:
df['error']=df.error.astype(int)

In [45]:
total_errors = df[df['Variation'].isin(['Test', 'Control'])]['error'].sum()
total_errors

np.int64(58503)

In [46]:
df.isna().sum()

Unnamed: 0               0
client_id                0
visitor_id               0
visit_id                 0
process_step             0
date_time                0
clnt_tenure_yr      300857
clnt_tenure_mnth    300857
clnt_age            300869
gendr               300857
num_accts           300857
bal                 300857
calls_6_mnth        300857
logons_6_mnth       300857
Variation           427406
step_num                 0
error                    0
duration            159112
duration_sec        159112
dtype: int64

In [47]:
df.date_time=pd.to_datetime(df['date_time'])
df.date_time

283823   2017-04-12 20:19:36
283822   2017-04-12 20:19:45
283821   2017-04-12 20:20:31
283820   2017-04-12 20:22:05
283819   2017-04-12 20:23:09
                 ...        
640076   2017-06-01 22:40:08
640075   2017-06-01 22:41:28
640074   2017-06-01 22:41:47
640073   2017-06-01 22:44:58
640072   2017-06-01 22:48:39
Name: date_time, Length: 744641, dtype: datetime64[us]

In [48]:
df['duration']=df.groupby((['client_id','visit_id'])).date_time.diff()
df.head(10)

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,step_num,error,duration,duration_sec
283823,283823,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,0,0,NaT,NaN
283822,283822,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,1,0,0 days 00:00:09,9.0
283821,283821,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,2,0,0 days 00:00:46,46.0
283820,283820,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,3,0,0 days 00:01:34,94.0
283819,283819,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,4,0,0 days 00:01:04,64.0
620520,620520,336,64757908_3400128256,649044751_80905125055_554468,start,2017-06-01 07:26:55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaT,NaN
620397,620397,336,64757908_3400128256,649044751_80905125055_554468,start,2017-06-01 07:42:43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0 days 00:15:48,948.0
439053,439053,546,475037402_89828530214,731811517_9330176838_94847,start,2017-06-17 10:03:29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaT,NaN
439052,439052,546,475037402_89828530214,731811517_9330176838_94847,step_1,2017-06-17 10:03:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0 days 00:00:10,10.0
439051,439051,546,475037402_89828530214,731811517_9330176838_94847,step_2,2017-06-17 10:03:52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,0 days 00:00:13,13.0


In [49]:
df['duration_sec'] = df['duration'].dt.total_seconds()

In [50]:
df.head()

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,step_num,error,duration,duration_sec
283823,283823,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,0,0,NaT,NaN
283822,283822,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,1,0,0 days 00:00:09,9.0
283821,283821,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,2,0,0 days 00:00:46,46.0
283820,283820,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,3,0,0 days 00:01:34,94.0
283819,283819,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,NaN,4,0,0 days 00:01:04,64.0


In [51]:
df.to_csv('../data/clean/clean_data_tableau.csv')